# Kiểm tra cấu trúc dữ liệu

Notebook này đưa các bước kiểm tra metadata, khảo sát loại văn bản, chọn tập văn bản, xem danh sách đã chọn và kiểm tra collection Qdrant vào một quy trình tuần tự. Chạy từng cell từ trên xuống dưới.

## 1. Khám phá dataset Hugging Face

Tải metadata để xem kích thước, tên cột, một vài dòng mẫu và phân bố loại/tình trạng văn bản. Sau đó đọc streaming hai bản ghi nội dung HTML để kiểm tra cấu trúc mà không tải toàn bộ nội dung.

In [1]:
import pandas as pd
from datasets import load_dataset

DS = "th1nhng0/vietnamese-legal-documents"
pd.set_option("display.max_colwidth", 80, "display.width", 200)

meta = load_dataset(DS, "metadata", split="data").to_pandas()
print("METADATA:", meta.shape, "\n", list(meta.columns), "\n")
display(meta.head(2).T)
for col in ["loai_van_ban", "tinh_trang_hieu_luc"]:
    print(f"--- {col} ---\n{meta[col].value_counts(dropna=False).head(15)}\n")
print("--- linh_vuc (mẫu) ---\n", meta["linh_vuc"].dropna().head(10).tolist(), "\n")

stream = load_dataset(DS, "content", split="data", streaming=True)
for i, row in enumerate(stream):
    print("CONTENT:", list(row.keys()), "id =", repr(row["id"]), "| len =", len(row["content_html"]))
    print(row["content_html"][:300].replace("\n", " "), "\n")
    if i >= 1:
        break

METADATA: (171556, 17) 
 ['id', 'title', 'so_ky_hieu', 'ngay_ban_hanh', 'loai_van_ban', 'ngay_co_hieu_luc', 'ngay_het_hieu_luc', 'nguon_thu_thap', 'ngay_dang_cong_bao', 'nganh', 'linh_vuc', 'co_quan_ban_hanh', 'chuc_danh', 'nguoi_ky', 'pham_vi', 'thong_tin_ap_dung', 'tinh_trang_hieu_luc'] 



,0,1
id,8733,105241
title,Thông tư số 01/GD-ĐT Hướng dẫn việc dạy học tiếng nói và chữ viết dân tộc th...,"Nghị quyết số 13/2014/NQ-HĐND Quy định mức thu, chế độ thu, nộp, quản lý và ..."
so_ky_hieu,01/GD-ĐT,13/2014/NQ-HĐND
ngay_ban_hanh,03/02/1997,05/12/2014
loai_van_ban,Thông tư,Nghị quyết
ngay_co_hieu_luc,03/02/1997,15/12/2014
ngay_het_hieu_luc,None,01/01/2017
nguon_thu_thap,Công báo điện tử;,None
ngay_dang_cong_bao,None,None
nganh,None,Tài chính


--- loai_van_ban ---
loai_van_ban
Quyết định                      91183
Nghị quyết                      28337
Thông tư                        17232
Bản dịch văn bản                10952
Chỉ thị                          8543
Nghị định                        5328
Thông tư liên tịch               3519
Văn bản hợp nhất                 2232
Sắc lệnh                          981
Văn bản hệ thống hóa              762
Công văn                          732
Luật                              610
Lệnh                              379
Văn bản hành chính liên quan      281
Pháp lệnh                         248
Name: count, dtype: int64

--- tinh_trang_hieu_luc ---
tinh_trang_hieu_luc
Hết hiệu lực toàn bộ       90721
Còn hiệu lực               71619
Hết hiệu lực một phần       5235
Chưa xác định               3205
Ngưng hiệu lực               373
Không còn phù hợp            255
Chưa có hiệu lực             146
Ngưng hiệu lực một phần        2
Name: count, dtype: int64

--- linh_vuc (mẫu) ---
 ['Chưa

## 2. Khảo sát Luật và Bộ luật

Tách metadata thành các nhóm Luật và Bộ luật; xem trạng thái, phạm vi và những tiêu đề có dấu hiệu là văn bản sửa đổi/bổ sung/bãi bỏ/đính chính. Bước này giúp hiểu nguồn dữ liệu trước khi chọn tập nạp.

In [2]:
meta["loai"] = meta["loai_van_ban"].fillna("").astype(str).str.strip()
meta["status"] = meta["tinh_trang_hieu_luc"].fillna("").astype(str).str.strip()

bo_luat = meta[meta["loai"].str.lower().eq("bộ luật")]
print("=== TẤT CẢ BỘ LUẬT TRONG DATASET ===")
display(bo_luat[["id", "so_ky_hieu", "title", "status", "pham_vi"]])
print("Tổng:", len(bo_luat))

luat = meta[meta["loai"].str.lower().eq("luật")].copy()
print("Tổng Luật trong dataset:", len(luat))
print("\nTheo trạng thái:")
display(luat["tinh_trang_hieu_luc"].fillna("").value_counts())
print("Theo phạm vi:")
display(luat["pham_vi"].fillna("").value_counts())

amend = luat[luat["title"].fillna("").astype(str).str.contains(
    "sửa đổi|bổ sung|bãi bỏ|đính chính", case=False, regex=True
)]
print("Số tiêu đề có từ khóa sửa đổi/bổ sung/bãi bỏ/đính chính:", len(amend))
display(amend[["so_ky_hieu", "title", "tinh_trang_hieu_luc"]].head(20))

=== TẤT CẢ BỘ LUẬT TRONG DATASET ===


,id,so_ky_hieu,title,status,pham_vi
19509,96115,92/2015/QH13,Bộ luật Tố tụng dân sự số 92/2015/QH13,Hết hiệu lực một phần,Trung ương
20477,96122,100/2015/QH13,Bộ luật Hình sự số 100/2015/QH13,Hết hiệu lực một phần,Trung ương
26650,9683,Không số,Bộ luật Dân sự số Không số,Hết hiệu lực toàn bộ,Trung ương
34281,18128,33/2005/QH11,Bộ luật Dân sự số 33/2005/QH11,Hết hiệu lực toàn bộ,Trung ương
36747,19411,19/2003/QH11,Bộ luật Tố tụng hình sự số 19/2003/QH11,Hết hiệu lực toàn bộ,Trung ương
50180,18131,40/2005/QH11,Bộ luật Hàng hải Việt Nam số 40/2005/QH11,Hết hiệu lực toàn bộ,Trung ương
67641,19482,24/2004/QH11,Bộ luật Tố tụng dân sự số 24/2004/QH11,Hết hiệu lực toàn bộ,Trung ương
77529,27615,10/2012/QH13,Bộ luật Lao động số 10/2012/QH13,Hết hiệu lực toàn bộ,Trung ương
86523,139264,45/2019/QH14,Bộ Luật lao động số 45/2019/QH14,Hết hiệu lực một phần,Trung ương
87075,2066,42-LCT/HĐNN8,Bộ luật Hàng hải số 42-LCT/HĐNN8,Hết hiệu lực toàn bộ,Trung ương


Tổng: 17
Tổng Luật trong dataset: 610

Theo trạng thái:


tinh_trang_hieu_luc
Hết hiệu lực toàn bộ     271
Còn hiệu lực             179
Hết hiệu lực một phần    154
Chưa có hiệu lực           6
Name: count, dtype: int64

Theo phạm vi:


pham_vi
Trung ương    572
Toàn quốc      35
                3
Name: count, dtype: int64

Số tiêu đề có từ khóa sửa đổi/bổ sung/bãi bỏ/đính chính: 166


,so_ky_hieu,title,tinh_trang_hieu_luc
713,48/2014/QH13,"Luật Sửa đổi, bổ sung một số điều của Luật Giao thông đường thủy nội địa số ...",Hết hiệu lực toàn bộ
886,71/2014/QH13,"Luật Sửa đổi, bổ sung một số điều của các luật về thuế số 71/2014/QH13",Hết hiệu lực toàn bộ
1023,44/2024/QH15,Luật Dược sửa đổi 2024 số 44/2024/QH15,Hết hiệu lực một phần
4217,61/2014/QH13,"Luật Sửa đổi, bổ sung một số điều của Luật Hàng không dân dụng Việt Nam số 6...",Hết hiệu lực một phần
5585,52/2024/QH15,"Luật sửa đổi, bổ sung một số điều của Luật Sĩ quan Quân đội nhân dân Việt Na...",Còn hiệu lực
7217,82/2025/QH15,"LUẬT SỬA ĐỔI, BỔ SUNG MỘT SỐ ĐIỀU CỦA LUẬT TỔ CHỨC VIỆN KIỂM SÁT NHÂN DÂN SỐ...",Còn hiệu lực
11726,76/2025/QH15,"Luật Sửa đổi, bổ sung một số điều của Luật Doanh nghiệp số 76/2025/QH15",Còn hiệu lực
11793,46/2005/QH11,"Luật sửa đổi, bổ sung một số điều của Luật Khoáng sản số 46/2005/QH11",Hết hiệu lực toàn bộ
13306,số 99/2025/QH15,"Luật sửa đổi, bổ sung một số điều của Bộ luật Tố tụng hình sự số số 99/2025/...",Còn hiệu lực
18531,65/2020/QH14,"Luật Sửa đổi, bổ sung một số điều của Luật Tổ chức Quốc hội số 65/2020/QH14",Hết hiệu lực một phần


## 3. Chọn tập văn bản

Lọc theo tầng loại văn bản, tình trạng hiệu lực, phạm vi, năm ban hành và từ khóa sửa đổi. Các tham số dưới đây thay cho đối số dòng lệnh của script để cell chạy được trực tiếp trong Jupyter. Kết quả được lưu vào `data/selected_meta.parquet` để các bước sau sử dụng.

Có thể sửa `SELECTED_TIERS`, `PHAM_VI`, `NO_LOCAL`, `TU_NAM` và `KEEP_AMENDMENTS` rồi chạy lại cell.

In [3]:
import os

TIERS = {
    "A": ["Bộ luật", "Luật", "Pháp lệnh", "Nghị định"],
    "B": ["Thông tư", "Thông tư liên tịch"],
    "C": ["Nghị quyết", "Quyết định", "Chỉ thị", "Văn bản hợp nhất"],
}
TRANG_THAI = ["Còn hiệu lực", "Hết hiệu lực một phần"]
KIEM_TRA = ["45/2019/QH14"]

# Tham số lựa chọn tương đương mặc định của select_all.py
SELECTED_TIERS = ["A", "B"]
PHAM_VI = "all"  # Ví dụ: "Trung ương", "Địa phương", "Toàn quốc"
NO_LOCAL = False
TU_NAM = 0  # 0 nghĩa là không lọc theo năm
KEEP_AMENDMENTS = False

loai2tier = {loai.lower(): tier for tier in SELECTED_TIERS for loai in TIERS[tier]}
m = pd.Series(True, index=meta.index)
print("=== Phiếu lọc ===")
print(f"{len(meta):>7} ban đầu")
buoc = [
    (f"Trạng thái thuộc {TRANG_THAI}", meta["tinh_trang_hieu_luc"].fillna("").astype(str).str.strip().isin(TRANG_THAI)),
    (f"Loại thuộc tầng {SELECTED_TIERS}", meta["loai_van_ban"].fillna("").astype(str).str.strip().str.lower().isin(loai2tier)),
]
if PHAM_VI != "all":
    buoc.append((f"Phạm vi = {PHAM_VI}", meta["pham_vi"].fillna("").astype(str).str.strip().eq(PHAM_VI)))
if NO_LOCAL:
    buoc.append(("Không thuộc phạm vi Địa phương", meta["pham_vi"].fillna("").astype(str).str.strip().ne("Địa phương")))
if TU_NAM:
    nam = pd.to_datetime(meta["ngay_ban_hanh"], format="%d/%m/%Y", errors="coerce").dt.year
    buoc.append((f"Ban hành từ năm {TU_NAM}", nam.isna() | (nam >= TU_NAM)))
if not KEEP_AMENDMENTS:
    buoc.append(("Loại tiêu đề văn bản sửa đổi/bổ sung/bãi bỏ/đính chính", ~meta["title"].fillna("").astype(str).str.contains("sửa đổi|bổ sung|bãi bỏ|đính chính", case=False, regex=True)))
for ten, dieu_kien in buoc:
    m &= dieu_kien
    print(f"{int(m.sum()):>7} sau điều kiện: {ten}")

sel = meta[m].copy()
sel["tier"] = sel["loai_van_ban"].fillna("").astype(str).str.strip().str.lower().map(loai2tier)
sel["_d"] = pd.to_datetime(sel["ngay_ban_hanh"], format="%d/%m/%Y", errors="coerce")
sel = sel.sort_values(["tier", "_d"], ascending=[True, False]).drop(columns="_d")

print(f"\n=== ĐÃ CHỌN {len(sel)} văn bản ===")
display(sel.groupby(["tier", "loai_van_ban"]).size().rename("số văn bản"))
print("Theo trạng thái:")
display(sel["tinh_trang_hieu_luc"].value_counts())
trung = sel["so_ky_hieu"].fillna("").str.strip()
n_trung = int(trung[trung != ""].duplicated(keep=False).sum())
print(f"Số hiệu trùng trong tập chọn: {n_trung}")
for so in KIEM_TRA:
    co = sel[trung == so]
    print(f"Kiểm tra {so}: " + (f"CÓ ({', '.join(co['id'].astype(str))})" if len(co) else "KHÔNG có trong tập chọn"))

os.makedirs("data", exist_ok=True)
sel.to_parquet("data/selected_meta.parquet", index=False)
print("Đã lưu data/selected_meta.parquet")

=== Phiếu lọc ===
 171556 ban đầu
  76854 sau điều kiện: Trạng thái thuộc ['Còn hiệu lực', 'Hết hiệu lực một phần']
  14286 sau điều kiện: Loại thuộc tầng ['A', 'B']
  11724 sau điều kiện: Loại tiêu đề văn bản sửa đổi/bổ sung/bãi bỏ/đính chính

=== ĐÃ CHỌN 11724 văn bản ===


tier  loai_van_ban      
A     Bộ luật                  6
      Luật                   246
      Nghị định             2259
      Pháp lệnh               44
B     Thông tư              7424
      Thông tư liên tịch    1745
Name: số văn bản, dtype: int64

Theo trạng thái:


tinh_trang_hieu_luc
Còn hiệu lực             10075
Hết hiệu lực một phần     1649
Name: count, dtype: int64

Số hiệu trùng trong tập chọn: 941
Kiểm tra 45/2019/QH14: CÓ (139264)
Đã lưu data/selected_meta.parquet


## 4. Xem danh sách đã chọn

Đọc file Parquet vừa tạo và hiển thị số hiệu, loại văn bản và tiêu đề để rà soát nhanh tập đầu vào.

In [4]:
pd.set_option("display.width", 250, "display.max_colwidth", 110)
sel = pd.read_parquet("data/selected_meta.parquet")
display(sel[["so_ky_hieu", "loai_van_ban", "title"]])

,so_ky_hieu,loai_van_ban,title
0,280/2026/NĐ-CP,Nghị định,"Nghị định số 280/2026/NĐ-CP Quy định về tổ chức, hoạt động và quản lý, đầu tư vốn nhà nước, giám sát, đánh..."
1,279/2026/NĐ-CP,Nghị định,"Nghị định số 279/2026/NĐ-CP Quy định chức năng, nhiệm vụ, quyền hạn và cơ cấu tổ chức của Bộ Giáo dục và Đ..."
2,272/2026/NĐ-CP,Nghị định,"Nghị định số 272/2026/NĐ-CP Quy định chi tiết một số điều và biện pháp để tổ chức, hướng dẫn thi hành Nghị..."
3,265/2026/NĐ-CP,Nghị định,"Nghị định số 265/2026/NĐ-CP Quy định xử phạt vi phạm hành chính trong lĩnh vực tiết kiệm, chống lãng phí"
4,266/2026/NĐ-CP,Nghị định,"Nghị định số 266/2026/NĐ-CP Quy định chi tiết và hướng dẫn thi hành một số điều của Luật Tiết kiệm, chống ..."
...,...,...,...
11719,825-VP,Thông tư,Thông tư số 825-VP Về thuế đặc biệt đánh vào xe hơi vận tải
11720,60-NV/CC,Thông tư,Thông tư số 60-NV/CC Về việc thi hành Sắc lệnh số 54 ngày 3 tháng 11 năm 1945 ấn định những điều kiện cho ...
11721,52-NV/cC,Thông tư,Thông tư số 52-NV/cC Về ciệc thi hành Sắc lệnh ngày 03 tháng 11 năm 1945 ấn định những điều kiện cho công ...
11722,47/VBHN-BTC,Thông tư,"Quy định về thủ tục thu, nộp tiền phạt, biên lai thu tiền phạt và kinh phí từ ngân sách nhà nước bảo đảm h..."


## 5. Kiểm tra collection Qdrant 

Kết nối collection đã nạp và thống kê tình trạng văn bản, loại văn bản, tầng, cấp chunk, số văn bản, chunk dài nhất và point thiếu vector sparse. Cell này yêu cầu Qdrant đang chạy và collection đã được nạp dữ liệu.

In [9]:
import sys
from pathlib import Path

ROOT_DIR = Path(r"D:\RAG")

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from collections import Counter
from ingestion.qdrant_db import COLLECTION, SPARSE, get_client

client = get_client()
if not client.collection_exists(COLLECTION):
    raise RuntimeError(f"Chưa có collection '{COLLECTION}'. Hãy tạo collection và chạy ingest trước.")
total = client.count(COLLECTION, exact=True).count
print(f"Collection '{COLLECTION}' | tổng points: {total}")
if total == 0:
    raise RuntimeError("Collection đang trống. Hãy nạp dữ liệu trước.")

status, documents, levels, kinds, tiers = (Counter() for _ in range(5))
missing_sparse = max_length = 0
offset = None
while True:
    points, offset = client.scroll(
        COLLECTION, limit=500, offset=offset,
        with_payload=["so_hieu", "tinh_trang", "cap", "noi_dung", "loai", "tier", "src_id"],
        with_vectors=[SPARSE],
    )
    for point in points:
        payload = point.payload
        status[payload.get("tinh_trang")] += 1
        documents[payload.get("src_id")] += 1
        levels[payload.get("cap")] += 1
        kinds[payload.get("loai")] += 1
        tiers[payload.get("tier")] += 1
        max_length = max(max_length, len(payload.get("noi_dung") or ""))
        missing_sparse += not point.vector or SPARSE not in point.vector
    if offset is None:
        break

print("Theo tình trạng:", dict(status))
print("Theo loại văn bản:", dict(kinds))
print("Theo tầng:", dict(tiers))
print("Theo cấp chunk:", dict(levels))
print("Số văn bản:", len(documents))
print("Văn bản nhiều point nhất (src_id):", documents.most_common(5))
print("Chunk dài nhất:", max_length, "ký tự")
print("Point thiếu vector sparse:", missing_sparse)

Collection 'luat_vn' | tổng points: 35519
Theo tình trạng: {'mot_phan': 19012, 'con_hieu_luc': 16507}
Theo loại văn bản: {'Luật': 26178, 'Bộ luật': 3235, 'Nghị định': 4707, 'Pháp lệnh': 1399}
Theo tầng: {'A': 35519}
Theo cấp chunk: {'dieu': 21452, 'khoan': 14067}
Số văn bản: 386
Văn bản nhiều point nhất (src_id): [('95942', 695), ('96122', 652), ('96115', 628), ('96172', 612), ('023a9ab0-58eb-11f1-b20f-3bfcb2ea511c', 537)]
Chunk dài nhất: 2955 ký tự
Point thiếu vector sparse: 0
